# 01 — Rapport global AP-HP

Ce notebook génère le rapport HTML pour l'ensemble du groupe AP-HP.

**Sortie** : `output/rapport_global_aphp.html`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

## Chargement des données

In [ ]:
import pandas as pd

aphp = pd.read_csv(DATA_DIR / 'aphp_data.csv')
reg  = pd.read_csv(DATA_DIR / 'regional_data.csv')

print(f"AP-HP  : {len(aphp):,} lignes | années {sorted(aphp.annee.unique())}")
print(f"Régional: {len(reg):,} lignes")

# Aperçu global AP-HP
aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == 'TOTAL')]

## Visualisations interactives

In [ ]:
from chart_utils import (
    line_evolution, bar_comparison, stacked_treatments,
    donut_market_share, heatmap_appareils, waterfall_trends,
    regional_comparison, TREATMENT_COLS, GHU_LIST
)

aphp_total = aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == 'TOTAL')]
ghu_total  = aphp[(aphp.entite.isin(GHU_LIST)) & (aphp.appareil == 'TOTAL')]

# Évolution patients
fig = line_evolution(aphp_total, 'annee', 'nb_patients', 'entite',
                     'Évolution du nombre de patients — AP-HP')
fig.show()

In [ ]:
# Waterfall variation annuelle
fig_wf = waterfall_trends(aphp_total, 'Variation annuelle — AP-HP')
fig_wf.show()

In [ ]:
# Séjours par mode de PEC
import pandas as pd

melted = aphp_total.melt(
    id_vars=['annee'], value_vars=list(TREATMENT_COLS.keys()),
    var_name='type', value_name='nb_sejours')
melted['label'] = melted['type'].map(TREATMENT_COLS)

fig_s = line_evolution(melted, 'annee', 'nb_sejours', 'label',
                       'Séjours par mode de prise en charge — AP-HP',
                       entities=list(TREATMENT_COLS.values()))
fig_s.show()

In [ ]:
# Parts de marché GHU (dernière année)
last_year = int(aphp_total.annee.max())
ghu_last = ghu_total[ghu_total.annee == last_year]

fig_d = donut_market_share(ghu_last, 'entite', 'nb_patients',
                           f'Répartition par GHU — {last_year}')
fig_d.show()

In [ ]:
# Évolution du nombre de patients par GHU
fig_ghu = line_evolution(ghu_total, 'annee', 'nb_patients', 'entite',
                         'Patients par GHU — évolution', entities=GHU_LIST)
fig_ghu.show()

In [ ]:
# Heatmap par appareil (AP-HP)
fig_h = heatmap_appareils(aphp, 'AP-HP', title='Évolution par appareil (index, moy=1) — AP-HP')
fig_h.show()

In [ ]:
# Contexte régional
reg_total = reg[(reg.appareil == 'TOTAL')]
fig_reg = regional_comparison(reg_total, 'nb_patients',
                              'Patients — AP-HP vs contexte régional')
fig_reg.show()

## Génération du rapport HTML

In [ ]:
from report_builder import build_rapport_global

out = build_rapport_global(DATA_DIR, OUTPUT_DIR)
print(f"Rapport généré : {out}")

In [ ]:
# Affichage du rapport dans le notebook (iframe)
from IPython.display import IFrame
IFrame(str(out.relative_to(Path.cwd().parent)), width='100%', height=800)